In [ ]:
import os
import numpy as np
import pandas as pd
import mne
import mne_bids
from pathlib import Path

# -----------------------------
# Dataset information
# -----------------------------
bids_dir = r"E:\IEEG-FMRI Dataset"

subjects = mne_bids.get_entity_vals(bids_dir, "subject")

session = "iemu"
datatype = "ieeg"
task = "film"
acquisition = "clinical"

summary = []

# -----------------------------
# Loop through all subjects
# -----------------------------
for subject in subjects:

    print(f"\nProcessing subject {subject}")

    # Skip subjects without iEEG session
    iemu_folder = Path(bids_dir) / f"sub-{subject}" / "ses-iemu"

    if not iemu_folder.exists():
        print(f"Skipping subject {subject}: no iEEG session")
        continue

    # ------------------------------------
    # Read channels.tsv
    # ------------------------------------
    channels_path = mne_bids.BIDSPath(
        subject=subject,
        session=session,
        datatype=datatype,
        task=task,
        acquisition=acquisition,
        suffix="channels",
        extension=".tsv",
        root=bids_dir,
    )

    # print(subject)
    # print(channels_path)
    # print(channels_path.match())

    channels = pd.read_csv(channels_path.match()[0], sep="\t")

    # ------------------------------------
    # Read BrainVision recording
    # ------------------------------------
    data_path = mne_bids.BIDSPath(
        subject=subject,
        session=session,
        datatype=datatype,
        task=task,
        acquisition=acquisition,
        suffix="ieeg",
        extension=".vhdr",
        root=bids_dir,
    )

    raw = mne.io.read_raw_brainvision(
        str(data_path.match()[0]),
        preload=True,
        verbose=False,
    )

    # ------------------------------------
    # Keep only EEG / ECoG / SEEG channels
    # ------------------------------------
    raw.set_channel_types(
        {
            ch_name: str(ch_type).lower()
            if str(ch_type).lower() in ["ecog", "seeg", "eeg"]
            else "misc"
            for ch_name, ch_type in zip(
                raw.ch_names,
                channels["type"].values
            )
        }
    )

    raw.drop_channels(
        [
            ch
            for ch, typ in zip(raw.ch_names, raw.get_channel_types())
            if typ == "misc"
        ]
    )

    # ------------------------------------
    # Remove bad/noisy channels
    # ------------------------------------
    bad_channels = channels.loc[
        channels["status"] == "bad",
        "name"
    ].tolist()

    bad_channels = [ch for ch in bad_channels if ch in raw.ch_names]

    raw.drop_channels(bad_channels)

    # ------------------------------------
    # Remove 50 Hz line noise and its harmonics
    # ------------------------------------
    raw.notch_filter(freqs=np.arange(50, 251, 50))

    # ------------------------------------
    # Common Average Reference
    # ------------------------------------
    raw_car, _ = mne.set_eeg_reference(raw.copy(), "average")

    # Remove invalid measurement date
    raw_car.set_meas_date(None)

    save_dir = Path(r"E:\ieeg-fmri-dataset-analysis\preprocess")
    save_dir.mkdir(exist_ok=True)

    raw_car.save(
        save_dir / f"sub-{subject}_{task}_preprocessed_raw.fif",
        overwrite=True
    )

    # ------------------------------------
    # Store
    # ------------------------------------

    summary.append({
    "Subject": subject,
    "Sampling_Frequency": raw_car.info["sfreq"],
    "Channels_Remaining": len(raw_car.ch_names)
    })

    print(f"Sampling frequency: {raw_car.info['sfreq']} Hz")
    print(f"Remaining channels: {len(raw_car.ch_names)}")

summary_df = pd.DataFrame(summary)
summary_df


Processing subject 01


C:\Users\muqadasah\AppData\Local\Temp\ipykernel_17888\1217013557.py:79: RuntimeWarning: The unit for channel(s) AH+, ECG+, EMG+, MKR1+, MKR2+, ORB+, abdo+, thor+ has changed from V to NA.
  raw.set_channel_types(


Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 13517 samples (6.600 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.8s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    3.5s


sEEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('sEEG',) reference.
Writing E:\ieeg-fmri-dataset-analysis\preprocess\sub-01_film_preprocessed_raw.fif
Closing E:\ieeg-fmri-dataset-analysis\preprocess\sub-01_film_preprocessed_raw.fif
[done]
Sampling frequency: 2048.0 Hz
Remaining channels: 101

Processing subject 02


...

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.7s


ECoG channel type selected for re-referencing
Applying average reference.
Applying a custom ('ECoG',) reference.
Writing E:\ieeg-fmri-dataset-analysis\preprocess\sub-61_film_preprocessed_raw.fif
Closing E:\ieeg-fmri-dataset-analysis\preprocess\sub-61_film_preprocessed_raw.fif
[done]
Sampling frequency: 2048.0 Hz
Remaining channels: 58

Processing subject 62
Skipping subject 62: no iEEG session

Processing subject 63
Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 3381 samples (6.604 s)



C:\Users\muqadasah\AppData\Local\Temp\ipykernel_17888\1217013557.py:79: RuntimeWarning: The unit for channel(s) AH+, ECG+, EMG+, MKR1+, MKR2+, Orb+ has changed from V to NA.
  raw.set_channel_types(
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


ECoG channel type selected for re-referencing
Applying average reference.
Applying a custom ('ECoG',) reference.
Writing E:\ieeg-fmri-dataset-analysis\preprocess\sub-63_film_preprocessed_raw.fif
Closing E:\ieeg-fmri-dataset-analysis\preprocess\sub-63_film_preprocessed_raw.fif
[done]
Sampling frequency: 512.0 Hz
Remaining channels: 70


,Subject,Sampling_Frequency,Channels_Remaining
0,01,2048.0,101
1,02,512.0,59
2,03,512.0,101
3,05,2048.0,73
4,06,512.0,102
5,07,512.0,55
6,09,2048.0,69
7,10,512.0,90
8,12,2048.0,64
9,13,512.0,94


In [ ]:
# #PREPROCESSING FOR REST FILES

# import os
# import numpy as np
# import pandas as pd
# import mne
# import mne_bids
# from pathlib import Path

# # -----------------------------
# # Dataset information
# # -----------------------------
# bids_dir = r"E:\IEEG-FMRI Dataset"

# subjects = mne_bids.get_entity_vals(bids_dir, "subject")

# session = "iemu"
# datatype = "ieeg"
# task = "rest"
# acquisition = "clinical"

# summary = []

# # -----------------------------
# # Loop through all subjects
# # -----------------------------
# for subject in subjects:

#     print(f"\nProcessing subject {subject}")

#     # Skip subjects without iEEG session
#     iemu_folder = Path(bids_dir) / f"sub-{subject}" / "ses-iemu"

#     if not iemu_folder.exists():
#         print(f"Skipping subject {subject}: no iEEG session")
#         continue

#     # ------------------------------------
#     # Read channels.tsv
#     # ------------------------------------
#     channels_path = mne_bids.BIDSPath(
#         subject=subject,
#         session=session,
#         datatype=datatype,
#         task=task,
#         acquisition=acquisition,
#         suffix="channels",
#         extension=".tsv",
#         root=bids_dir,
#     )

#     print(subject)
#     print(channels_path)
#     print(channels_path.match())


#     channels = pd.read_csv(channels_path.match()[0], sep="\t")

#     # ------------------------------------
#     # Read BrainVision recording
#     # ------------------------------------
#     data_path = mne_bids.BIDSPath(
#         subject=subject,
#         session=session,
#         datatype=datatype,
#         task=task,
#         acquisition=acquisition,
#         suffix="ieeg",
#         extension=".vhdr",
#         root=bids_dir,
#     )

#     raw = mne.io.read_raw_brainvision(
#         str(data_path.match()[0]),
#         preload=True,
#         verbose=False,
#     )

#     # ------------------------------------
#     # Keep only EEG / ECoG / SEEG channels
#     # ------------------------------------
#     raw.set_channel_types(
#         {
#             ch_name: str(ch_type).lower()
#             if str(ch_type).lower() in ["ecog", "seeg", "eeg"]
#             else "misc"
#             for ch_name, ch_type in zip(
#                 raw.ch_names,
#                 channels["type"].values
#             )
#         }
#     )

#     raw.drop_channels(
#         [
#             ch
#             for ch, typ in zip(raw.ch_names, raw.get_channel_types())
#             if typ == "misc"
#         ]
#     )

#     # ------------------------------------
#     # Remove bad/noisy channels
#     # ------------------------------------
#     bad_channels = channels.loc[
#         channels["status"] == "bad",
#         "name"
#     ].tolist()

#     bad_channels = [ch for ch in bad_channels if ch in raw.ch_names]

#     raw.drop_channels(bad_channels)

#     # ------------------------------------
#     # Remove 50 Hz line noise and its harmonics
#     # ------------------------------------
#     raw.notch_filter(freqs=np.arange(50, 251, 50))

#     # ------------------------------------
#     # Common Average Reference
#     # ------------------------------------
#     raw_car, _ = mne.set_eeg_reference(raw.copy(), "average")

#     # Remove invalid measurement date
#     raw_car.set_meas_date(None)

#     save_dir = Path(r"E:\ieeg-fmri-dataset-analysis\preprocess")
#     save_dir.mkdir(exist_ok=True)

#     raw_car.save(
#         save_dir / f"sub-{subject}_{task}_preprocessed_raw.fif",
#         overwrite=True
#     )

#     # ------------------------------------
#     # Store
#     # ------------------------------------

#     summary.append({
#     "Subject": subject,
#     "Sampling_Frequency": raw_car.info["sfreq"],
#     "Channels_Remaining": len(raw_car.ch_names)
#     })

#     print(f"Sampling frequency: {raw_car.info['sfreq']} Hz")
#     print(f"Remaining channels: {len(raw_car.ch_names)}")

# summary_df = pd.DataFrame(summary)
# summary_df

#SUB-33 DOESN'T HAVE A REST FILE. AND THE ABOVE CODE STOPS AT SUB-33. SO WE FIX THE ISSUE


Processing subject 01
01
E:/IEEG-FMRI Dataset/sub-01/ses-iemu/ieeg/sub-01_ses-iemu_task-rest_acq-clinical_channels.tsv
[BIDSPath(
root: E:/IEEG-FMRI Dataset
datatype: ieeg
basename: sub-01_ses-iemu_task-rest_acq-clinical_run-1_channels.tsv)]


C:\Users\muqadasah\AppData\Local\Temp\ipykernel_17888\3641657625.py:82: RuntimeWarning: The unit for channel(s) AH+, ECG+, EMG+, MKR1+, MKR2+, ORB+, abdo+, thor+ has changed from V to NA.
  raw.set_channel_types(


Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 13517 samples (6.600 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    1.4s


sEEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('sEEG',) reference.
Overwriting existing file.
Writing E:\ieeg-fmri-dataset-analysis\preprocess\sub-01_rest_preprocessed_raw.fif
Closing E:\ieeg-fmri-dataset-analysis\preprocess\sub-01_rest_preprocessed_raw.fif
[done]
Sampling frequency: 2048.0 Hz
Remaining channels: 102

Processing subject 02
02
E:/IEEG-FMRI Dataset/sub-02/ses-iemu/ieeg/sub-02_ses-iemu_task-rest_acq-clinical_channels.tsv
[BIDSPath(
root: E:/IEEG-FMRI Dataset
datatype: ieeg
basename: sub-02_ses-iemu_task-rest_acq-clinical_run-1_channels.tsv)]
Filtering raw data in 1 contiguous segment
Setting up band-stop filter



...

[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    0.2s


Overwriting existing file.
Writing E:\ieeg-fmri-dataset-analysis\preprocess\sub-31_rest_preprocessed_raw.fif
Closing E:\ieeg-fmri-dataset-analysis\preprocess\sub-31_rest_preprocessed_raw.fif
[done]
Sampling frequency: 512.0 Hz
Remaining channels: 96

Processing subject 32
32
E:/IEEG-FMRI Dataset/sub-32/ses-iemu/ieeg/sub-32_ses-iemu_task-rest_acq-clinical_channels.tsv
[BIDSPath(
root: E:/IEEG-FMRI Dataset
datatype: ieeg
basename: sub-32_ses-iemu_task-rest_acq-clinical_run-1_channels.tsv)]
Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 13517 samples (6.600 s)



C:\Users\muqadasah\AppData\Local\Temp\ipykernel_17888\3641657625.py:82: RuntimeWarning: The unit for channel(s) AH+, ECG+, EMG+, MKR+, abdo+, thor+ has changed from V to NA.
  raw.set_channel_types(
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.3s


ECoG channel type selected for re-referencing
Applying average reference.
Applying a custom ('ECoG',) reference.
Overwriting existing file.
Writing E:\ieeg-fmri-dataset-analysis\preprocess\sub-32_rest_preprocessed_raw.fif
Closing E:\ieeg-fmri-dataset-analysis\preprocess\sub-32_rest_preprocessed_raw.fif
[done]
Sampling frequency: 2048.0 Hz
Remaining channels: 43

Processing subject 33
33
E:/IEEG-FMRI Dataset/sub-33/ses-iemu/ieeg/sub-33_ses-iemu_task-rest_acq-clinical_channels.tsv
[]


IndexError: list index out of range

In [4]:
#PREPROCESSING FOR REST FILES

import os
import numpy as np
import pandas as pd
import mne
import mne_bids
from pathlib import Path

# -----------------------------
# Dataset information
# -----------------------------
bids_dir = r"E:\IEEG-FMRI Dataset"

subjects = mne_bids.get_entity_vals(bids_dir, "subject")

session = "iemu"
datatype = "ieeg"
task = "rest"
acquisition = "clinical"

summary = []

# -----------------------------
# Loop through all subjects
# -----------------------------
for subject in subjects:

    print(f"\nProcessing subject {subject}")

    # Skip subjects without iEEG session
    iemu_folder = Path(bids_dir) / f"sub-{subject}" / "ses-iemu"

    if not iemu_folder.exists():
        print(f"Skipping subject {subject}: no iEEG session")
        continue

    # ------------------------------------
    # Read channels.tsv
    # ------------------------------------
    ieeg_dir = Path(bids_dir) / f"sub-{subject}" / f"ses-{session}" / datatype

    channel_files = sorted(
        ieeg_dir.glob(f"*task-{task}_acq-clinical*_channels.tsv")
    )

    if len(channel_files) == 0:
        print(f"No channels file found for subject {subject}")
        continue

    channels = pd.read_csv(channel_files[0], sep="\t")

    # ------------------------------------
    # Read BrainVision recording
    # ------------------------------------
    vhdr_files = sorted(
    ieeg_dir.glob(f"*task-{task}_acq-clinical*_ieeg.vhdr")
)

    if len(vhdr_files) == 0:
        print(f"No recording found for subject {subject}")
        continue

    raw = mne.io.read_raw_brainvision(
        str(vhdr_files[0]),
        preload=True,
        verbose=False,
    )

    # ------------------------------------
    # Keep only EEG / ECoG / SEEG channels
    # ------------------------------------
    raw.set_channel_types(
        {
            ch_name: str(ch_type).lower()
            if str(ch_type).lower() in ["ecog", "seeg", "eeg"]
            else "misc"
            for ch_name, ch_type in zip(
                raw.ch_names,
                channels["type"].values
            )
        }
    )

    raw.drop_channels(
        [
            ch
            for ch, typ in zip(raw.ch_names, raw.get_channel_types())
            if typ == "misc"
        ]
    )

    # ------------------------------------
    # Remove bad/noisy channels
    # ------------------------------------
    bad_channels = channels.loc[
        channels["status"] == "bad",
        "name"
    ].tolist()

    bad_channels = [ch for ch in bad_channels if ch in raw.ch_names]

    raw.drop_channels(bad_channels)

    # ------------------------------------
    # Remove 50 Hz line noise and its harmonics
    # ------------------------------------
    raw.notch_filter(freqs=np.arange(50, 251, 50))

    # ------------------------------------
    # Common Average Reference
    # ------------------------------------
    raw_car, _ = mne.set_eeg_reference(raw.copy(), "average")

    # Remove invalid measurement date
    raw_car.set_meas_date(None)

    save_dir = Path(r"E:\ieeg-fmri-dataset-analysis\preprocess")
    save_dir.mkdir(exist_ok=True)

    raw_car.save(
        save_dir / f"sub-{subject}_{task}_preprocessed_raw.fif",
        overwrite=True
    )

    # ------------------------------------
    # Store
    # ------------------------------------

    summary.append({
    "Subject": subject,
    "Sampling_Frequency": raw_car.info["sfreq"],
    "Channels_Remaining": len(raw_car.ch_names)
    })

    print(f"Sampling frequency: {raw_car.info['sfreq']} Hz")
    print(f"Remaining channels: {len(raw_car.ch_names)}")

summary_df = pd.DataFrame(summary)
summary_df


Processing subject 01


C:\Users\muqadasah\AppData\Local\Temp\ipykernel_17888\1264894985.py:73: RuntimeWarning: The unit for channel(s) AH+, ECG+, EMG+, MKR1+, MKR2+, ORB+, abdo+, thor+ has changed from V to NA.
  raw.set_channel_types(


Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 13517 samples (6.600 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    1.4s


sEEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('sEEG',) reference.
Overwriting existing file.
Writing E:\ieeg-fmri-dataset-analysis\preprocess\sub-01_rest_preprocessed_raw.fif
Closing E:\ieeg-fmri-dataset-analysis\preprocess\sub-01_rest_preprocessed_raw.fif
[done]
Sampling frequency: 2048.0 Hz
Remaining channels: 102

Processing subject 02


...

C:\Users\muqadasah\AppData\Local\Temp\ipykernel_17888\1264894985.py:73: RuntimeWarning: The unit for channel(s) AH+, ECG+, EMG+, MKR1+, MKR2+, abdo+, orb+, thor+ has changed from V to NA.
  raw.set_channel_types(
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.2s


ECoG channel type selected for re-referencing
Applying average reference.
Applying a custom ('ECoG',) reference.
Writing E:\ieeg-fmri-dataset-analysis\preprocess\sub-61_rest_preprocessed_raw.fif
Closing E:\ieeg-fmri-dataset-analysis\preprocess\sub-61_rest_preprocessed_raw.fif
[done]
Sampling frequency: 2048.0 Hz
Remaining channels: 58

Processing subject 62
Skipping subject 62: no iEEG session

Processing subject 63
Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 3381 samples (6.604 s)



C:\Users\muqadasah\AppData\Local\Temp\ipykernel_17888\1264894985.py:73: RuntimeWarning: The unit for channel(s) AH+, ECG+, EMG+, MKR1+, MKR2+, Orb+ has changed from V to NA.
  raw.set_channel_types(
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


ECoG channel type selected for re-referencing
Applying average reference.
Applying a custom ('ECoG',) reference.
Writing E:\ieeg-fmri-dataset-analysis\preprocess\sub-63_rest_preprocessed_raw.fif
Closing E:\ieeg-fmri-dataset-analysis\preprocess\sub-63_rest_preprocessed_raw.fif
[done]
Sampling frequency: 512.0 Hz
Remaining channels: 70


,Subject,Sampling_Frequency,Channels_Remaining
0,01,2048.0,102
1,02,2048.0,60
2,03,512.0,101
3,05,2048.0,75
4,06,512.0,101
5,07,512.0,55
6,09,2048.0,69
7,10,512.0,90
8,12,2048.0,64
9,13,512.0,94
